# Entrenamiento axial final por split de paciente

Notebook reproducible para preparar el entrenamiento final del modelo axial `axial_t2_alkafri`.
Por seguridad inicia con `RUN_MODE = "preflight"`: valida entorno, datos, labels, split, duplicados y forward sin entrenar.

No publica artifacts, no reemplaza el baseline axial historico, no accede a recursos cloud y no constituye validacion clinica.


## Modos

- `preflight`: no entrena; valida entorno, paths, inventario, imagenes, mascaras, labels y split.
- `smoke`: subconjunto pequeno deterministico, maximo dos epochs, artifacts con `smoke_only=true`.
- `full`: split completo, seleccion por validation, test held-out una sola vez y paquete local candidato/final.


In [ ]:
import importlib.util
import os
import subprocess
import sys

PFI_INSTALL_NOTEBOOK_DEPS = os.getenv("PFI_INSTALL_NOTEBOOK_DEPS", "1") == "1"
REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "pydicom": "pydicom",
    "SimpleITK": "SimpleITK",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing and PFI_INSTALL_NOTEBOOK_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
elif missing:
    raise RuntimeError(f"Dependencias faltantes: {missing}")


In [ ]:
from __future__ import annotations

import csv
import dataclasses
import hashlib
import json
import math
import os
import platform
import random
import re
import subprocess
import tempfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Literal

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    import pydicom
except Exception:
    pydicom = None
try:
    import SimpleITK as sitk
except Exception:
    sitk = None
try:
    import nibabel as nib
except Exception:
    nib = None


## Configuracion central


In [ ]:
def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    return default if raw is None else raw.strip().lower() in {"1", "true", "yes", "on"}


def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


PFI_CLOUD_MODE = env_bool("PFI_CLOUD_MODE", False)
PFI_USE_GOOGLE_DRIVE = env_bool("PFI_USE_GOOGLE_DRIVE", True)
PFI_SYNC_DRY_RUN = env_bool("PFI_SYNC_DRY_RUN", True)
DRIVE_ROOT = Path(os.getenv("PFI_DRIVE_ROOT", "/content/drive/MyDrive/PFI_MVP"))
EXTERNAL_ROOT = DRIVE_ROOT if PFI_USE_GOOGLE_DRIVE else Path(os.getenv("PFI_LOCAL_EXTERNAL_ROOT", "/tmp/pfi_mvp"))


@dataclass(frozen=True)
class TrainConfig:
    RUN_MODE: Literal["preflight", "smoke", "full"] = os.getenv("RUN_MODE", "preflight")
    SEED: int = int(os.getenv("PFI_SEED", "2026"))
    RUN_ID: str = os.getenv("PFI_RUN_ID", "axial-final-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
    REPO_ROOT: Path = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))
    DATASET_ROOT: Path = Path(os.getenv("PFI_DATASET_ROOT", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    IMAGES_ROOT: Path = Path(os.getenv("AXIAL_IMAGES_DIR", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    MASKS_ROOT: Path = Path(os.getenv("AXIAL_MASKS_DIR", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    SPLIT_MANIFEST_PATH: Path = Path(os.getenv("AXIAL_E9_CURATED_SPLIT_CSV", str(EXTERNAL_ROOT / "results" / "E9_alkafri_axial_t2_final_labels_baseline" / "E9_t2_final_labels_curated_split.csv")))
    OUTPUT_ROOT: Path = Path(os.getenv("PFI_OUTPUT_ROOT", str(EXTERNAL_ROOT / "outputs" / "axial_final_training")))
    RESUME_ROOT: Path = Path(os.getenv("PFI_RESUME_ROOT", str(EXTERNAL_ROOT / "outputs" / "axial_final_training" / "resume")))
    TARGET_SIZE: tuple[int, int] = (256, 256)
    NUM_CLASSES: int = 6
    BASE_CHANNELS: int = int(os.getenv("PFI_BASE_CHANNELS", "16"))
    BATCH_SIZE: int = int(os.getenv("PFI_BATCH_SIZE", "8"))
    NUM_WORKERS: int = int(os.getenv("PFI_NUM_WORKERS", "0"))
    MAX_EPOCHS: int = int(os.getenv("PFI_MAX_EPOCHS", "80"))
    EARLY_STOPPING_PATIENCE: int = int(os.getenv("PFI_EARLY_STOP_PATIENCE", "12"))
    LEARNING_RATE: float = float(os.getenv("PFI_LR", "0.0008"))
    WEIGHT_DECAY: float = float(os.getenv("PFI_WEIGHT_DECAY", "0.0001"))
    USE_AMP: bool = env_bool("PFI_USE_AMP", True)
    GRADIENT_CLIP_NORM: float = float(os.getenv("PFI_GRADIENT_CLIP_NORM", "1.0"))
    CHECKPOINT_EVERY_EPOCH: bool = env_bool("PFI_CHECKPOINT_EVERY_EPOCH", True)
    ALLOW_SPLIT_GENERATION: bool = env_bool("ALLOW_SPLIT_GENERATION", False)
    ENABLE_TRAIN_AUGMENTATION: bool = env_bool("ENABLE_TRAIN_AUGMENTATION", True)
    QUALITY_GATE_DICE_MACRO_FOREGROUND: float = float(os.getenv("QUALITY_GATE_DICE_MACRO_FOREGROUND", "0.70"))
    QUALITY_GATE_REQUIRE_PATIENT_HELDOUT: bool = True
    QUALITY_GATE_REQUIRE_RAW0_REPORT: bool = True
    QUALITY_GATE_REQUIRE_IOU: bool = True
    RESUME_MODE: Literal["off", "auto", "required"] = os.getenv("RESUME_MODE", "auto")
    AXIAL_IMAGE_COL: str = os.getenv("AXIAL_IMAGE_COL", "image_file_path")
    AXIAL_MASK_COL: str = os.getenv("AXIAL_MASK_COL", "final_label_file_path")
    AXIAL_PATIENT_COL: str = os.getenv("AXIAL_PATIENT_COL", "case_id_norm")
    AXIAL_SPLIT_COL: str = os.getenv("AXIAL_SPLIT_COL", "split")
    AXIAL_STUDY_COL: str = os.getenv("AXIAL_STUDY_COL", "case_id_norm")
    SMOKE_MAX_RECORDS_PER_SPLIT: int = int(os.getenv("SMOKE_MAX_RECORDS_PER_SPLIT", "12"))
    WEIGHT_MAX_RECORDS: int = int(os.getenv("PFI_WEIGHT_MAX_RECORDS", "256"))
    RAW0_WEIGHT_BOOST: float = float(os.getenv("AXIAL_RAW0_WEIGHT_BOOST", "3.0"))

    def __post_init__(self):
        if self.RUN_MODE not in {"preflight", "smoke", "full"}:
            raise ValueError(f"RUN_MODE invalido: {self.RUN_MODE}")
        if self.TARGET_SIZE != (256, 256) or self.NUM_CLASSES != 6 or self.BASE_CHANNELS != 16:
            raise ValueError("Arquitectura axial final debe ser target 256, numClasses 6, baseChannels 16")


CFG = TrainConfig()
MODEL_KEY = "axial_t2_alkafri"
RAW_ALLOWED_LABELS = {0, 50, 100, 150, 200, 250}
RAW_TO_CLASS_INDEX = {250: 0, 0: 1, 50: 2, 100: 3, 150: 4, 200: 5}
CLASS_INDEX_TO_NAME = {0: "background_250", 1: "raw_0", 2: "raw_50", 3: "raw_100", 4: "raw_150", 5: "raw_200"}
FOREGROUND_CLASS_IDS = [1, 2, 3, 4, 5]
FOREGROUND_EXCLUDING_RAW0 = [2, 3, 4, 5]
humanReviewRequired = True
notClinicalDiagnosis = True
OUTPUT_DIRS = {name: CFG.OUTPUT_ROOT / CFG.RUN_ID / name for name in ["models", "resume", "manifests", "metrics", "figures", "predictions", "logs", "reports"]}
print(json.dumps(dataclasses.asdict(CFG), indent=2, default=str))


## Entorno y reproducibilidad


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def git_value(args: list[str]) -> str:
    try:
        return subprocess.check_output(["git", *args], cwd=str(CFG.REPO_ROOT), text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unavailable"


def environment_manifest() -> dict[str, Any]:
    cuda = torch.cuda.is_available()
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cudaAvailable": cuda,
        "gpuName": torch.cuda.get_device_name(0) if cuda else None,
        "gpuVramGb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda else None,
        "gitCommit": git_value(["rev-parse", "HEAD"]),
        "gitDirty": bool(git_value(["status", "--short"])),
        "PFI_CLOUD_MODE": PFI_CLOUD_MODE,
        "PFI_USE_GOOGLE_DRIVE": PFI_USE_GOOGLE_DRIVE,
        "PFI_SYNC_DRY_RUN": PFI_SYNC_DRY_RUN,
    }


set_seed(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if CFG.RUN_MODE == "full" and DEVICE.type != "cuda":
    raise RuntimeError("RUN_MODE=full requiere CUDA para evitar una corrida final accidental en CPU")
ENV_MANIFEST = environment_manifest()
print(json.dumps(ENV_MANIFEST, indent=2))


## Utilidades seguras


In [ ]:
def mkdirs_for_run() -> None:
    for path in OUTPUT_DIRS.values():
        path.mkdir(parents=True, exist_ok=True)


def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_bytes(path: Path, data: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(handle, "wb") as fh:
            fh.write(data)
            fh.flush()
            os.fsync(fh.fileno())
        Path(tmp_name).replace(path)
    except Exception:
        Path(tmp_name).unlink(missing_ok=True)
        raise


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    atomic_write_bytes(path, json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8"))


def atomic_write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        atomic_write_bytes(path, b"")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(handle, "w", encoding="utf-8", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
            fh.flush()
            os.fsync(fh.fileno())
        Path(tmp_name).replace(path)
    except Exception:
        Path(tmp_name).unlink(missing_ok=True)
        raise


def safe_source_name(path: str | Path) -> str:
    return Path(path).name


## Lectura, preprocesamiento y labels


In [ ]:
def open_medical_array(path: Path) -> np.ndarray:
    suffixes = [s.lower() for s in path.suffixes]
    suffix = path.suffix.lower()
    if suffix == ".npy":
        return np.asarray(np.load(path))
    if suffix in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
        return np.asarray(Image.open(path))
    if suffix in {".dcm", ".ima"}:
        if pydicom is None:
            raise RuntimeError("pydicom no disponible")
        return np.asarray(pydicom.dcmread(str(path)).pixel_array)
    if suffix in {".mha", ".mhd"}:
        if sitk is None:
            raise RuntimeError("SimpleITK no disponible")
        return sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    if suffix == ".nii" or suffixes[-2:] == [".nii", ".gz"]:
        if nib is None:
            raise RuntimeError("nibabel no disponible")
        return np.asarray(nib.load(str(path)).get_fdata())
    raise ValueError(f"Formato no soportado: {suffix}")


def as_2d_slice(array: np.ndarray, case_id: str) -> np.ndarray:
    arr = np.asarray(array)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3 and 1 in arr.shape:
        return np.squeeze(arr)
    raise ValueError(f"{case_id}: shape axial no soportado {arr.shape}")


def robust_normalize(image: np.ndarray) -> np.ndarray:
    arr = np.nan_to_num(np.asarray(image, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [1.0, 99.0])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(arr.min()), float(arr.max())
    if hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    return ((np.clip(arr, lo, hi) - lo) / (hi - lo)).astype(np.float32)


def resize_image(image: np.ndarray) -> np.ndarray:
    return np.asarray(Image.fromarray(image.astype(np.float32)).resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.BILINEAR), dtype=np.float32)


def resize_mask(mask: np.ndarray) -> np.ndarray:
    return np.asarray(Image.fromarray(mask.astype(np.int32)).resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.NEAREST), dtype=np.int64)


def apply_label_mapping(mask: np.ndarray, case_id: str) -> np.ndarray:
    labels = {int(v) for v in np.unique(mask)}
    unexpected = sorted(labels - RAW_ALLOWED_LABELS)
    if unexpected:
        raise ValueError(f"{case_id}: labels inesperados {unexpected}")
    mapped = np.zeros(mask.shape, dtype=np.int64)
    for raw_label, class_id in RAW_TO_CLASS_INDEX.items():
        mapped[np.asarray(mask) == raw_label] = class_id
    return mapped


## Inventario y split por paciente


In [ ]:
@dataclass(frozen=True)
class AxialRecord:
    image_path: str
    mask_path: str
    patientId: str
    studyId: str
    sliceId: str
    split: str
    sourceImage: str
    sourceMask: str


def resolve_path(value: Any, bases: Iterable[Path]) -> Path:
    raw = Path(str(value))
    if raw.is_absolute() and raw.exists():
        return raw
    for base in bases:
        for candidate in [base / raw, base / raw.name]:
            if candidate.exists():
                return candidate
    return raw


def slice_id_from_path(path: Path) -> str:
    return "slice_" + hashlib.sha256(path.name.encode("utf-8")).hexdigest()[:12]


def build_records_from_split_manifest() -> list[AxialRecord]:
    if not CFG.SPLIT_MANIFEST_PATH.exists():
        if CFG.ALLOW_SPLIT_GENERATION:
            raise NotImplementedError("Solo se permite generar candidate split con revision humana posterior")
        raise FileNotFoundError(f"Falta split manifest E9: {CFG.SPLIT_MANIFEST_PATH}")
    df = pd.read_csv(CFG.SPLIT_MANIFEST_PATH)
    required = [CFG.AXIAL_IMAGE_COL, CFG.AXIAL_MASK_COL, CFG.AXIAL_PATIENT_COL, CFG.AXIAL_SPLIT_COL]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"Faltan columnas: {missing}")
    bases = [CFG.DATASET_ROOT, CFG.IMAGES_ROOT, CFG.MASKS_ROOT, CFG.SPLIT_MANIFEST_PATH.parent]
    records = []
    for row in df.itertuples(index=False):
        image = resolve_path(getattr(row, CFG.AXIAL_IMAGE_COL), bases)
        mask = resolve_path(getattr(row, CFG.AXIAL_MASK_COL), bases)
        if not image.exists() or not mask.exists():
            raise FileNotFoundError(f"Archivo axial inexistente: {image.name} / {mask.name}")
        patient = str(getattr(row, CFG.AXIAL_PATIENT_COL)).strip()
        split = str(getattr(row, CFG.AXIAL_SPLIT_COL)).strip().lower()
        study = str(getattr(row, CFG.AXIAL_STUDY_COL)).strip() if hasattr(row, CFG.AXIAL_STUDY_COL) else patient
        records.append(AxialRecord(str(image), str(mask), patient, study, slice_id_from_path(image), split, safe_source_name(image), safe_source_name(mask)))
    return records


def summarize_records(records: list[AxialRecord]) -> pd.DataFrame:
    df = pd.DataFrame([dataclasses.asdict(r) for r in records])
    return df.groupby("split").agg(n_slices=("sliceId", "count"), n_patients=("patientId", "nunique"))


In [ ]:
def validate_split_integrity(records: list[AxialRecord]) -> dict[str, Any]:
    df = pd.DataFrame([dataclasses.asdict(r) for r in records])
    if set(df["split"]) - {"train", "val", "test"}:
        raise ValueError(f"Splits invalidos: {sorted(set(df['split']))}")
    for split in ["train", "val", "test"]:
        if int((df["split"] == split).sum()) == 0:
            raise ValueError(f"Split vacio: {split}")
    for key in ["patientId", "studyId", "image_path", "mask_path"]:
        leaked = df.groupby(key)["split"].nunique()
        leaked = leaked[leaked > 1]
        if not leaked.empty:
            raise ValueError(f"Leakage cross-split en {key}: {len(leaked)}")
    return {"patientHeldout": True, "splitCounts": df["split"].value_counts().to_dict()}


def scan_labels_and_shapes(records: list[AxialRecord]) -> dict[str, Any]:
    label_counts: Counter[int] = Counter()
    raw0_by_split: dict[str, set[str]] = defaultdict(set)
    formats: Counter[str] = Counter()
    dimensions: Counter[str] = Counter()
    for index, record in enumerate(records, 1):
        image = as_2d_slice(open_medical_array(Path(record.image_path)), record.sliceId)
        mask = as_2d_slice(open_medical_array(Path(record.mask_path)), record.sliceId)
        if image.shape != mask.shape:
            raise ValueError(f"{record.sliceId}: image/mask shape mismatch")
        labels = {int(v) for v in np.unique(mask)}
        unexpected = sorted(labels - RAW_ALLOWED_LABELS)
        if unexpected:
            raise ValueError(f"{record.sliceId}: labels inesperados {unexpected}")
        label_counts.update(labels)
        if 0 in labels:
            raw0_by_split[record.split].add(record.patientId)
        formats[Path(record.image_path).suffix.lower()] += 1
        dimensions[str(tuple(image.shape))] += 1
        if index % 250 == 0:
            print(f"scan {index}/{len(records)}")
    return {"labelCounts": dict(label_counts), "raw0PatientsBySplit": {k: len(v) for k, v in raw0_by_split.items()}, "formats": dict(formats), "dimensions": dict(dimensions)}


## Duplicados y leakage por hash


In [ ]:
def compute_duplicate_report(records: list[AxialRecord], full_hash: bool) -> pd.DataFrame:
    rows = []
    for record in records:
        for kind, raw_path in [("image", record.image_path), ("mask", record.mask_path)]:
            path = Path(raw_path)
            digest = sha256_stream(path) if full_hash else f"size:{path.stat().st_size}"
            rows.append({"kind": kind, "sha256": digest, "patientId": record.patientId, "split": record.split, "source": safe_source_name(path)})
    return pd.DataFrame(rows)


def validate_no_duplicate_leakage(duplicate_df: pd.DataFrame) -> dict[str, Any]:
    problems = []
    for (kind, digest), group in duplicate_df.groupby(["kind", "sha256"]):
        if group["split"].nunique() > 1 or group["patientId"].nunique() > 1:
            problems.append({"kind": kind, "sha256": digest, "splits": group["split"].unique().tolist()})
    if problems:
        raise ValueError(f"Duplicados/leakage detectados: {problems[:10]}")
    duplicate_df.to_csv(OUTPUT_DIRS["metrics"] / "duplicate_report.csv", index=False)
    return {"duplicateLeakage": False, "checkedRows": int(len(duplicate_df))}


## Dataset, DataLoader y sanity check


In [ ]:
def maybe_augment(image: np.ndarray, mask: np.ndarray, rng: random.Random) -> tuple[np.ndarray, np.ndarray]:
    if not CFG.ENABLE_TRAIN_AUGMENTATION:
        return image, mask
    if rng.random() < 0.5:
        image, mask = np.flip(image, 1).copy(), np.flip(mask, 1).copy()
    if rng.random() < 0.15:
        k = rng.choice([-1, 1])
        image, mask = np.rot90(image, k).copy(), np.rot90(mask, k).copy()
    if rng.random() < 0.25:
        image = np.clip(image * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05), 0, 1).astype(np.float32)
    return image, mask


class AxialSegmentationDataset(Dataset):
    def __init__(self, records: list[AxialRecord], split: str, augment: bool):
        self.records = [r for r in records if r.split == split]
        self.augment = augment
        if not self.records:
            raise ValueError(f"Dataset vacio: {split}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.records[index]
        image = resize_image(robust_normalize(as_2d_slice(open_medical_array(Path(record.image_path)), record.sliceId)))
        mask = apply_label_mapping(resize_mask(as_2d_slice(open_medical_array(Path(record.mask_path)), record.sliceId)), record.sliceId)
        if self.augment:
            image, mask = maybe_augment(image, mask, random.Random(CFG.SEED + index))
        return {"image": torch.from_numpy(image[None].astype(np.float32)), "mask": torch.from_numpy(mask.astype(np.int64)), "patientId": record.patientId, "studyId": record.studyId, "sliceId": record.sliceId, "sourceImage": record.sourceImage, "sourceMask": record.sourceMask}


def limit_records_for_smoke(records: list[AxialRecord]) -> list[AxialRecord]:
    return sum(([r for r in records if r.split == split][:CFG.SMOKE_MAX_RECORDS_PER_SPLIT] for split in ["train", "val", "test"]), [])


def seed_worker(worker_id: int) -> None:
    random.seed(CFG.SEED + worker_id)
    np.random.seed(CFG.SEED + worker_id)


def build_loaders(records: list[AxialRecord], smoke: bool) -> dict[str, DataLoader]:
    if smoke:
        records = limit_records_for_smoke(records)
    generator = torch.Generator().manual_seed(CFG.SEED)
    return {split: DataLoader(AxialSegmentationDataset(records, split, split == "train"), batch_size=CFG.BATCH_SIZE, shuffle=split == "train", num_workers=CFG.NUM_WORKERS, worker_init_fn=seed_worker, generator=generator, pin_memory=DEVICE.type == "cuda", persistent_workers=CFG.NUM_WORKERS > 0, drop_last=False) for split in ["train", "val", "test"]}


## Arquitectura AxialUNet2D compatible con runtime


In [ ]:
sys.path.insert(0, str(CFG.REPO_ROOT / "ai_service"))
from pfi_ai_service.model_architectures import AxialUNet2D, build_checkpoint_model, checkpoint_state_dict


def build_model() -> nn.Module:
    return AxialUNet2D(num_classes=CFG.NUM_CLASSES, base_channels=CFG.BASE_CHANNELS)


def strict_runtime_artifact_smoke(path: Path) -> list[int]:
    checkpoint = torch.load(str(path), map_location="cpu", weights_only=False)
    model, _metadata = build_checkpoint_model(MODEL_KEY, checkpoint)
    model.eval()
    with torch.inference_mode():
        output = model(torch.randn(1, 1, *CFG.TARGET_SIZE))
    expected = [1, CFG.NUM_CLASSES, *CFG.TARGET_SIZE]
    if list(output.shape) != expected:
        raise ValueError(f"Shape runtime invalida: {list(output.shape)}")
    return list(output.shape)


## Loss y metricas Dice/IoU


In [ ]:
def estimate_class_weights(records: list[AxialRecord]) -> torch.Tensor:
    counts = np.ones(CFG.NUM_CLASSES, dtype=np.float64)
    for record in [r for r in records if r.split == "train"][:CFG.WEIGHT_MAX_RECORDS]:
        mapped = apply_label_mapping(resize_mask(as_2d_slice(open_medical_array(Path(record.mask_path)), record.sliceId)), record.sliceId)
        counts += np.bincount(mapped.reshape(-1), minlength=CFG.NUM_CLASSES)
    weights = 1.0 / np.sqrt(counts / counts.sum())
    weights = weights / weights.mean()
    weights[1] *= CFG.RAW0_WEIGHT_BOOST
    return torch.tensor(np.clip(weights, 0.25, 8.0), dtype=torch.float32)


def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor, include_background: bool = False, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.softmax(logits, dim=1)
    one_hot = F.one_hot(target, CFG.NUM_CLASSES).permute(0, 3, 1, 2).float()
    class_ids = range(CFG.NUM_CLASSES) if include_background else FOREGROUND_CLASS_IDS
    losses = []
    for class_id in class_ids:
        p, t = probs[:, class_id], one_hot[:, class_id]
        losses.append(1 - ((2 * (p * t).sum((1, 2)) + eps) / (p.sum((1, 2)) + t.sum((1, 2)) + eps)))
    return torch.stack(losses).mean()


def multiclass_loss(logits: torch.Tensor, target: torch.Tensor, ce: nn.Module) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    ce_loss = ce(logits, target)
    dice_loss_value = soft_dice_loss(logits, target)
    return ce_loss + dice_loss_value, ce_loss, dice_loss_value


def dice_iou_for_class(pred: np.ndarray, truth: np.ndarray, class_id: int) -> tuple[float | None, float | None, int, int]:
    pred_mask, truth_mask = pred == class_id, truth == class_id
    tp = int(np.logical_and(pred_mask, truth_mask).sum())
    fp = int(np.logical_and(pred_mask, ~truth_mask).sum())
    fn = int(np.logical_and(~pred_mask, truth_mask).sum())
    dice_den, iou_den = 2 * tp + fp + fn, tp + fp + fn
    return (None if dice_den == 0 else 2 * tp / dice_den, None if iou_den == 0 else tp / iou_den, int(truth_mask.any()), int(pred_mask.any()))


def metrics_from_logits(logits: torch.Tensor, target: torch.Tensor) -> dict[str, Any]:
    pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
    truth = target.detach().cpu().numpy()
    per_class = {}
    for class_id, name in CLASS_INDEX_TO_NAME.items():
        values = [dice_iou_for_class(p, t, class_id) for p, t in zip(pred, truth)]
        dice = [v[0] for v in values if v[0] is not None]
        iou = [v[1] for v in values if v[1] is not None]
        per_class[name] = {"dice": float(np.mean(dice)) if dice else None, "iou": float(np.mean(iou)) if iou else None, "gt_present_cases": sum(v[2] for v in values), "pred_present_cases": sum(v[3] for v in values)}
    fg_dice = [per_class[CLASS_INDEX_TO_NAME[c]]["dice"] for c in FOREGROUND_CLASS_IDS if per_class[CLASS_INDEX_TO_NAME[c]]["dice"] is not None]
    fg_iou = [per_class[CLASS_INDEX_TO_NAME[c]]["iou"] for c in FOREGROUND_CLASS_IDS if per_class[CLASS_INDEX_TO_NAME[c]]["iou"] is not None]
    excl = [per_class[CLASS_INDEX_TO_NAME[c]]["dice"] for c in FOREGROUND_EXCLUDING_RAW0 if per_class[CLASS_INDEX_TO_NAME[c]]["dice"] is not None]
    return {"perClass": per_class, "dice_macro_foreground": float(np.mean(fg_dice)) if fg_dice else None, "iou_macro_foreground": float(np.mean(fg_iou)) if fg_iou else None, "dice_macro_excluding_raw0": float(np.mean(excl)) if excl else None, "raw0Dice": per_class["raw_0"]["dice"], "raw0Iou": per_class["raw_0"]["iou"]}


## Entrenamiento, resume y checkpoints


In [ ]:
def save_checkpoint(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    os.close(handle)
    try:
        torch.save(payload, tmp_name)
        Path(tmp_name).replace(path)
    except Exception:
        Path(tmp_name).unlink(missing_ok=True)
        raise


def checkpoint_payload(model: nn.Module, optimizer: torch.optim.Optimizer, scheduler: Any, scaler: Any, epoch: int, best_metric: float, class_weights: torch.Tensor, smoke_only: bool) -> dict[str, Any]:
    return {"schemaVersion": "axial-final-training-v1", "modelKey": MODEL_KEY, "modelVersion": CFG.RUN_ID, "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "optimizerStateDict": optimizer.state_dict(), "schedulerStateDict": scheduler.state_dict(), "scalerStateDict": scaler.state_dict() if scaler else None, "epoch": epoch, "bestValidationMetric": best_metric, "trainingConfig": dataclasses.asdict(CFG), "labelMapping": {"rawToClassIndex": RAW_TO_CLASS_INDEX, "classIndexToName": CLASS_INDEX_TO_NAME}, "num_classes": CFG.NUM_CLASSES, "base_channels": CFG.BASE_CHANNELS, "target_size": CFG.TARGET_SIZE, "class_weights": class_weights.detach().cpu().tolist(), "gitCommit": ENV_MANIFEST.get("gitCommit"), "smoke_only": smoke_only, "humanReviewRequired": humanReviewRequired, "notClinicalDiagnosis": notClinicalDiagnosis}


def load_resume_if_allowed(model: nn.Module, optimizer: torch.optim.Optimizer, scheduler: Any, scaler: Any) -> tuple[int, float]:
    path = OUTPUT_DIRS["resume"] / "axial_t2_alkafri_final.last_checkpoint.pt"
    if CFG.RESUME_MODE == "off":
        return 0, -math.inf
    if not path.exists():
        if CFG.RESUME_MODE == "required":
            raise FileNotFoundError(path)
        return 0, -math.inf
    checkpoint = torch.load(str(path), map_location=DEVICE, weights_only=False)
    if checkpoint.get("modelKey") != MODEL_KEY or checkpoint.get("labelMapping", {}).get("rawToClassIndex") != RAW_TO_CLASS_INDEX:
        raise ValueError("Resume incompatible con modelKey o label mapping")
    model.load_state_dict(checkpoint_state_dict(checkpoint), strict=True)
    optimizer.load_state_dict(checkpoint["optimizerStateDict"])
    scheduler.load_state_dict(checkpoint["schedulerStateDict"])
    if scaler and checkpoint.get("scalerStateDict"):
        scaler.load_state_dict(checkpoint["scalerStateDict"])
    return int(checkpoint.get("epoch", 0)), float(checkpoint.get("bestValidationMetric", -math.inf))


In [ ]:
def aggregate_epoch_metrics(items: list[dict[str, Any]], loss: float, ce_loss: float, dice_loss_value: float) -> dict[str, Any]:
    return {"loss": loss, "cross_entropy": ce_loss, "dice_loss": dice_loss_value, "dice_macro_foreground": float(np.nanmean([x["dice_macro_foreground"] for x in items])), "iou_macro_foreground": float(np.nanmean([x["iou_macro_foreground"] for x in items])), "dice_macro_excluding_raw0": float(np.nanmean([x["dice_macro_excluding_raw0"] for x in items])), "raw0Dice": float(np.nanmean([x["raw0Dice"] for x in items]))}


def run_epoch(model: nn.Module, loader: DataLoader, ce: nn.Module, optimizer: torch.optim.Optimizer | None, scaler: Any | None) -> dict[str, Any]:
    training = optimizer is not None
    model.train(training)
    losses = ce_losses = dice_losses = 0.0
    metric_items = []
    for batch in loader:
        image, mask = batch["image"].to(DEVICE), batch["mask"].to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
            logits = model(image)
            loss, ce_loss, dice_loss_value = multiclass_loss(logits, mask, ce)
        if training:
            if scaler:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRADIENT_CLIP_NORM)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRADIENT_CLIP_NORM); optimizer.step()
        losses += float(loss.detach().cpu()); ce_losses += float(ce_loss.detach().cpu()); dice_losses += float(dice_loss_value.detach().cpu())
        metric_items.append(metrics_from_logits(logits, mask))
    n = max(len(loader), 1)
    return aggregate_epoch_metrics(metric_items, losses / n, ce_losses / n, dice_losses / n)


def train_model(records: list[AxialRecord], smoke_only: bool) -> dict[str, Any]:
    mkdirs_for_run()
    loaders = build_loaders(records, smoke_only)
    class_weights = estimate_class_weights(records).to(DEVICE)
    model = build_model().to(DEVICE)
    ce = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.USE_AMP and DEVICE.type == "cuda")
    start_epoch, best_val = load_resume_if_allowed(model, optimizer, scheduler, scaler)
    max_epochs = min(CFG.MAX_EPOCHS, 2) if smoke_only else CFG.MAX_EPOCHS
    history = []
    patience_left = CFG.EARLY_STOPPING_PATIENCE
    for epoch in range(start_epoch + 1, max_epochs + 1):
        train_metrics = run_epoch(model, loaders["train"], ce, optimizer, scaler)
        with torch.inference_mode():
            val_metrics = run_epoch(model, loaders["val"], ce, None, None)
        monitor = float(val_metrics["dice_macro_foreground"])
        scheduler.step(monitor)
        improved = monitor > best_val
        best_val = max(best_val, monitor)
        patience_left = CFG.EARLY_STOPPING_PATIENCE if improved else patience_left - 1
        row = {"epoch": epoch, "learning_rate": float(optimizer.param_groups[0]["lr"]), **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
        history.append(row)
        payload = checkpoint_payload(model, optimizer, scheduler, scaler, epoch, best_val, class_weights, smoke_only)
        save_checkpoint(OUTPUT_DIRS["resume"] / "axial_t2_alkafri_final.last_checkpoint.pt", payload)
        if improved:
            save_checkpoint(OUTPUT_DIRS["resume"] / "axial_t2_alkafri_final.best_checkpoint.pt", payload)
        atomic_write_csv(OUTPUT_DIRS["metrics"] / "axial_final_training_history.csv", history)
        if not smoke_only and patience_left <= 0:
            break
    return {"bestValidationMetric": best_val, "history": history}


## Evaluacion final, quality gate, manifest y model card


In [ ]:
def evaluate_test_once(records: list[AxialRecord]) -> dict[str, Any]:
    marker = OUTPUT_DIRS["metrics"] / "axial_final_test_evaluated_once.json"
    if marker.exists() and os.getenv("TEST_RERUN_CONFIRMATION") != CFG.RUN_ID:
        raise RuntimeError("Test held-out ya evaluado para este RUN_ID")
    loaders = build_loaders(records, False)
    checkpoint = torch.load(str(OUTPUT_DIRS["resume"] / "axial_t2_alkafri_final.best_checkpoint.pt"), map_location=DEVICE, weights_only=False)
    model = build_model().to(DEVICE)
    model.load_state_dict(checkpoint_state_dict(checkpoint), strict=True)
    model.eval()
    ce = nn.CrossEntropyLoss(weight=torch.tensor(checkpoint["class_weights"], dtype=torch.float32, device=DEVICE))
    with torch.inference_mode():
        result = run_epoch(model, loaders["test"], ce, None, None)
    result.update({"testEvaluatedOnce": True, "evaluatedAtUtc": utc_now()})
    atomic_write_json(marker, result)
    return result


def quality_gate(test_metrics: dict[str, Any], integrity: dict[str, Any]) -> dict[str, Any]:
    reasons = []
    if not integrity.get("patientHeldout"):
        reasons.append("split_no_confirmado_por_paciente")
    if test_metrics.get("dice_macro_foreground") is None or float(test_metrics["dice_macro_foreground"]) < CFG.QUALITY_GATE_DICE_MACRO_FOREGROUND:
        reasons.append("dice_macro_foreground_below_threshold")
    if CFG.QUALITY_GATE_REQUIRE_IOU and test_metrics.get("iou_macro_foreground") is None:
        reasons.append("iou_macro_foreground_missing")
    if CFG.QUALITY_GATE_REQUIRE_RAW0_REPORT and test_metrics.get("raw0Dice") is None:
        reasons.append("raw_0_missing_report")
    return {"qualityGatePassed": not reasons, "thresholdDiceMacroForeground": CFG.QUALITY_GATE_DICE_MACRO_FOREGROUND, "reasons": reasons, "humanReviewRequired": humanReviewRequired, "notClinicalDiagnosis": notClinicalDiagnosis}


def runtime_artifact_payload(best_checkpoint_path: Path, gate: dict[str, Any]) -> dict[str, Any]:
    checkpoint = torch.load(str(best_checkpoint_path), map_location="cpu", weights_only=False)
    return {"model_state_dict": checkpoint_state_dict(checkpoint), "num_classes": CFG.NUM_CLASSES, "base_channels": CFG.BASE_CHANNELS, "target_size": CFG.TARGET_SIZE, "classes": [CLASS_INDEX_TO_NAME[i] for i in range(CFG.NUM_CLASSES)], "label_map": RAW_TO_CLASS_INDEX, "metrics": gate, "training_status": "validated_baseline" if gate["qualityGatePassed"] else "candidate_below_quality_gate"}


def generate_manifest_and_model_card(artifact_path: Path, test_metrics: dict[str, Any], gate: dict[str, Any]) -> None:
    manifest = {"modelKey": MODEL_KEY, "version": CFG.RUN_ID, "artifactFile": "axial_t2_alkafri_final_best.pt", "artifactSha256": sha256_stream(artifact_path), "dataset": "ALKAFRI/Sudirman axial T2 lumbar MRI", "datasetVersion": "derived_from_split_manifest", "splitManifestSha256": sha256_stream(CFG.SPLIT_MANIFEST_PATH), "task": "lumbar_mri_multiclass_segmentation", "inputPlane": "axial", "classes": [CLASS_INDEX_TO_NAME[i] for i in range(CFG.NUM_CLASSES)], "rawToClassIndex": {str(k): v for k, v in RAW_TO_CLASS_INDEX.items()}, "metrics": {"dice": test_metrics["dice_macro_foreground"], "iou": test_metrics["iou_macro_foreground"], "dicePerClass": {}, "iouPerClass": {}, "diceMacroExcludingRaw0": test_metrics.get("dice_macro_excluding_raw0"), "raw0Dice": test_metrics.get("raw0Dice")}, "trainingStatus": "validated_baseline" if gate["qualityGatePassed"] else "candidate_below_quality_gate", "qualityGate": gate, "heldOutConfirmedByPatient": True, "targetSize": list(CFG.TARGET_SIZE), "baseChannels": CFG.BASE_CHANNELS, "numClasses": CFG.NUM_CLASSES, "stateDictFormat": "checkpoint_dict_with_model_state_dict", "gitCommit": ENV_MANIFEST.get("gitCommit"), "seed": CFG.SEED, "humanReviewRequired": True, "notClinicalDiagnosis": True}
    atomic_write_json(OUTPUT_DIRS["manifests"] / "axial_t2_alkafri_final_best.pt.manifest.json", manifest)
    atomic_write_bytes(OUTPUT_DIRS["manifests"] / "axial_t2_alkafri_final_best.pt.modelcard.md", f"# Model Card - axial_t2_alkafri_final_best.pt\n\nLabels raw_* sin traduccion anatomica cerrada. Revision humana requerida. No diagnostico clinico.\n".encode("utf-8"))


## Ejecucion preflight/smoke/full


In [ ]:
def preflight() -> dict[str, Any]:
    mkdirs_for_run()
    records = build_records_from_split_manifest()
    print(summarize_records(records))
    split_integrity = validate_split_integrity(records)
    label_report = scan_labels_and_shapes(records)
    duplicate_report = validate_no_duplicate_leakage(compute_duplicate_report(records, full_hash=CFG.RUN_MODE == "full"))
    loaders = build_loaders(records, smoke=True)
    batch = next(iter(loaders["train"]))
    model = build_model().to(DEVICE)
    with torch.inference_mode():
        output = model(batch["image"].to(DEVICE))
    report = {"runMode": CFG.RUN_MODE, "nRecords": len(records), "splitIntegrity": split_integrity, "labelReport": label_report, "duplicateReport": duplicate_report, "batchShape": list(batch["image"].shape), "maskShape": list(batch["mask"].shape), "forwardShape": list(output.shape), "humanReviewRequired": humanReviewRequired, "notClinicalDiagnosis": notClinicalDiagnosis}
    atomic_write_json(OUTPUT_DIRS["metrics"] / "dataset_integrity_report.json", report)
    atomic_write_json(OUTPUT_DIRS["metrics"] / "split_integrity_report.json", split_integrity)
    return report


PREFLIGHT_REPORT = preflight()
TRAINING_RESULT = TEST_RESULT = QUALITY_GATE_RESULT = None

if CFG.RUN_MODE in {"smoke", "full"}:
    all_records = build_records_from_split_manifest()
    TRAINING_RESULT = train_model(all_records, smoke_only=CFG.RUN_MODE == "smoke")
    if CFG.RUN_MODE == "full":
        TEST_RESULT = evaluate_test_once(all_records)
        QUALITY_GATE_RESULT = quality_gate(TEST_RESULT, {"patientHeldout": True})
        runtime_name = "axial_t2_alkafri_final_best.pt" if QUALITY_GATE_RESULT["qualityGatePassed"] else "axial_t2_alkafri_final_candidate.pt"
        runtime_path = OUTPUT_DIRS["models"] / runtime_name
        save_checkpoint(runtime_path, runtime_artifact_payload(OUTPUT_DIRS["resume"] / "axial_t2_alkafri_final.best_checkpoint.pt", QUALITY_GATE_RESULT))
        strict_runtime_artifact_smoke(runtime_path)
        generate_manifest_and_model_card(runtime_path, TEST_RESULT, QUALITY_GATE_RESULT)

print(json.dumps({"preflight": PREFLIGHT_REPORT, "training": TRAINING_RESULT, "test": TEST_RESULT, "quality": QUALITY_GATE_RESULT}, indent=2, default=str))


## Visualizaciones y salidas planificadas

Durante la ejecucion se pueden generar distribucion de clases por split, curvas train/val, Dice por clase, IoU por clase, matriz de confusion y ejemplos deterministas de input/GT/prediccion/overlay.
Los ejemplos deben incluir casos medianos, peores, con `raw_0` y sin `raw_0`, siempre con IDs tecnicos desidentificados.
